# Credit Card Fraud Detection
### End-to-End Machine Learning & Deep Learning Pipeline

---

**Objective:** Build a fraud detection system using the Kaggle Credit Card Fraud Detection dataset. The dataset contains transactions made by European cardholders in September 2013, with only 0.17% being fraudulent — making this a highly imbalanced classification problem.

**Pipeline:**
1. Data Cleaning & Preprocessing
2. Exploratory Data Analysis (EDA)
3. PCA Analysis & Feature Importance
4. Feature Engineering & Scaling
5. Train-Test Split
6. SMOTE (on training data only)
7. Clustering Analysis
8. Model Training & Comparison
9. Hyperparameter Tuning
10. Evaluation & Results

**Dataset:** [Kaggle - Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Author:** Rachana | MSc Artificial Intelligence

---
## 1. Setup & Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import time

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from scipy import stats

# SMOTE for class imbalance
from imblearn.over_sampling import SMOTE

# Clustering
from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score

# ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization, LSTM,
                                     Conv1D, MaxPooling1D, Flatten, Input)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Hyperparameter Tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Evaluation
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, matthews_corrcoef,
                             confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve, auc)

# Explainability
import shap

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print('All libraries imported successfully!')

In [ ]:
# Load the dataset
df = pd.read_csv('Data/creditcard.csv')
print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head()

---
## 2. Data Cleaning & Preprocessing

In [ ]:
# Basic dataset info
print('Dataset Info:')
print('=' * 50)
df.info()
print('\nStatistical Summary:')
df.describe()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum().sum())
print('No missing values found!' if df.isnull().sum().sum() == 0 else 'Missing values detected!')

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates:,}')

if duplicates > 0:
    print(f'Removing {duplicates} duplicates...')
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'New shape: {df.shape}')

In [ ]:
# Check for infinite values
inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
print(f'Infinite values: {inf_count}')

if inf_count > 0:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(df.median(), inplace=True)
    print('Infinite values replaced with median.')

In [ ]:
# Class distribution
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(f'  Legitimate (0): {class_counts[0]:,} ({class_pct[0]:.3f}%)')
print(f'  Fraud (1):      {class_counts[1]:,} ({class_pct[1]:.3f}%)')
print(f'\nImbalance ratio: 1:{class_counts[0] // class_counts[1]}')
print('This is extremely imbalanced - accuracy alone will be misleading!')

In [ ]:
# Compare fraud vs legitimate statistics
print('Legitimate Transactions:')
display(df[df['Class'] == 0][['Amount', 'Time']].describe())

print('\nFraudulent Transactions:')
display(df[df['Class'] == 1][['Amount', 'Time']].describe())

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71', '#e74c3c']

# Bar chart
bars = axes[0].bar(['Legitimate', 'Fraud'], class_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Transaction Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'{count:,}', ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['Legitimate', 'Fraud'], autopct='%1.3f%%',
            colors=colors, startangle=90, explode=(0, 0.1), shadow=True)
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Transaction Amount distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
axes[0].hist(df[df['Class'] == 0]['Amount'], bins=100, alpha=0.7, color='#2ecc71', label='Legitimate', density=True)
axes[0].hist(df[df['Class'] == 1]['Amount'], bins=50, alpha=0.7, color='#e74c3c', label='Fraud', density=True)
axes[0].set_title('Transaction Amount Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Amount')
axes[0].set_xlim(0, 500)
axes[0].legend()

# Box plot
df_plot = df[['Amount', 'Class']].copy()
df_plot['Class'] = df_plot['Class'].map({0: 'Legitimate', 1: 'Fraud'})
sns.boxplot(data=df_plot, x='Class', y='Amount', palette=colors, ax=axes[1], showfliers=False)
axes[1].set_title('Amount by Class', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Legitimate avg amount: ${df[df["Class"]==0]["Amount"].mean():.2f}')
print(f'Fraud avg amount:      ${df[df["Class"]==1]["Amount"].mean():.2f}')

In [ ]:
# Time distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Time by class
axes[0].hist(df[df['Class'] == 0]['Time'], bins=100, alpha=0.7, color='#2ecc71', label='Legitimate', density=True)
axes[0].hist(df[df['Class'] == 1]['Time'], bins=100, alpha=0.7, color='#e74c3c', label='Fraud', density=True)
axes[0].set_title('Transaction Time Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time (seconds)')
axes[0].legend()

# Extract hour from time
df['Hour'] = (df['Time'] / 3600).astype(int) % 24

# Fraud by hour
fraud_by_hour = df[df['Class'] == 1].groupby('Hour').size()
axes[1].bar(fraud_by_hour.index, fraud_by_hour.values, color='#e74c3c', alpha=0.8)
axes[1].set_title('Fraud Transactions by Hour', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Fraud Count')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(20, 16))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Top features correlated with fraud
target_corr = corr_matrix['Class'].drop('Class').abs().sort_values(ascending=False)
print('Top 10 features correlated with fraud:')
for feat, val in target_corr.head(10).items():
    print(f'  {feat}: {corr_matrix.loc[feat, "Class"]:.4f}')

In [ ]:
# Violin plots for top discriminative features
top_features = target_corr.head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    temp = df[[feat, 'Class']].copy()
    temp['Class'] = temp['Class'].map({0: 'Legitimate', 1: 'Fraud'})
    sns.violinplot(data=temp, x='Class', y=feat, palette=colors, ax=axes[i], inner='box', cut=0)
    axes[i].set_title(feat, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Top 8 Discriminative Features', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Outlier analysis using IQR method
outlier_results = []
for col in ['Amount', 'Time'] + top_features[:6]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    outlier_results.append({'Feature': col, 'Outliers': outliers, 'Percentage': round(outliers / len(df) * 100, 2)})

print('Outlier Analysis (IQR Method):')
display(pd.DataFrame(outlier_results))
print('\nNote: Outliers in PCA features may indicate fraud patterns, so we keep them.')

---
## 4. PCA Analysis & Feature Importance

In [ ]:
# Variance analysis of PCA components (V1-V28 are already PCA-transformed)
pca_features = [f'V{i}' for i in range(1, 29)]
pca_variances = df[pca_features].var().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Variance per component
axes[0].bar(range(1, 29), pca_variances.values, color='#3498db', edgecolor='black', linewidth=0.5)
axes[0].set_title('Variance of PCA Components', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Component')
axes[0].set_ylabel('Variance')

# Cumulative variance
cum_var = np.cumsum(pca_variances.values / pca_variances.sum())
axes[1].plot(range(1, 29), cum_var, 'ro-')
axes[1].axhline(y=0.95, color='gray', linestyle='--', label='95% threshold')
axes[1].set_title('Cumulative Variance', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Components')
axes[1].set_ylabel('Cumulative Proportion')
axes[1].legend()

plt.tight_layout()
plt.show()

n_95 = np.argmax(cum_var >= 0.95) + 1
print(f'{n_95} components explain 95% of the variance.')

In [ ]:
# 2D PCA visualisation
all_features = pca_features + ['Amount', 'Time']
scaler_pca = StandardScaler()
X_pca_input = scaler_pca.fit_transform(df[all_features])

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca_2d.fit_transform(X_pca_input)

plt.figure(figsize=(10, 7))
plt.scatter(X_2d[df['Class'] == 0, 0], X_2d[df['Class'] == 0, 1],
            c='#2ecc71', alpha=0.1, s=3, label='Legitimate')
plt.scatter(X_2d[df['Class'] == 1, 0], X_2d[df['Class'] == 1, 1],
            c='#e74c3c', alpha=0.8, s=15, label='Fraud', edgecolors='black', linewidths=0.3)
plt.title('2D PCA Projection', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(markerscale=3)
plt.tight_layout()
plt.show()

In [ ]:
# 3D PCA visualisation (interactive)
pca_3d = PCA(n_components=3, random_state=RANDOM_STATE)
X_3d = pca_3d.fit_transform(X_pca_input)

# Sample for performance
legit_idx = np.random.choice(np.where(df['Class'] == 0)[0], 5000, replace=False)
fraud_idx = np.where(df['Class'] == 1)[0]
sample_idx = np.concatenate([legit_idx, fraud_idx])

fig = px.scatter_3d(x=X_3d[sample_idx, 0], y=X_3d[sample_idx, 1], z=X_3d[sample_idx, 2],
                    color=df['Class'].iloc[sample_idx].astype(str),
                    color_discrete_map={'0': '#2ecc71', '1': '#e74c3c'},
                    title='3D PCA Projection (Sampled)', opacity=0.6)
fig.update_layout(template='plotly_dark', width=900, height=600)
fig.show()

In [ ]:
# Feature importance using Mann-Whitney U test
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

importance = []
for col in all_features:
    stat, p_val = stats.mannwhitneyu(fraud[col], legit[col], alternative='two-sided')
    importance.append({'Feature': col, 'P-Value': p_val, 'Correlation': abs(corr_matrix.loc[col, 'Class'])})

importance_df = pd.DataFrame(importance).sort_values('Correlation', ascending=False)

plt.figure(figsize=(12, 6))
plt.barh(importance_df['Feature'].head(15)[::-1], importance_df['Correlation'].head(15)[::-1],
         color='#3498db', edgecolor='black')
plt.title('Feature Importance (Correlation with Fraud)', fontsize=14, fontweight='bold')
plt.xlabel('Absolute Correlation')
plt.tight_layout()
plt.show()

---
## 5. Feature Engineering & Scaling

In [ ]:
# Create new features
# Hour was already created in EDA
df['Amount_Log'] = np.log1p(df['Amount'])  # log transform to reduce skewness

print('New features created: Hour, Amount_Log')
print(f'Dataset shape: {df.shape}')

In [ ]:
# Define features and target
feature_cols = pca_features + ['Amount', 'Time', 'Hour', 'Amount_Log']

X = df[feature_cols].copy()
y = df['Class'].copy()

# Final check - make sure there are no NaN values
print(f'Features shape: {X.shape}')
print(f'NaN values: {X.isnull().sum().sum()}')
print(f'Target distribution: {y.value_counts().to_dict()}')

---
## 6. Train-Test Split

In [ ]:
# Split data (80% train, 20% test) with stratification to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train-Test Split:')
print(f'  Train: {X_train.shape[0]:,} samples')
print(f'  Test:  {X_test.shape[0]:,} samples')
print(f'\nTrain class distribution:')
print(f'  Legitimate: {(y_train == 0).sum():,} | Fraud: {(y_train == 1).sum():,}')
print(f'\nTest class distribution:')
print(f'  Legitimate: {(y_test == 0).sum():,} | Fraud: {(y_test == 1).sum():,}')

In [ ]:
# StandardScaler - fit on training data ONLY to prevent data leakage
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)

print('StandardScaler applied:')
print('  - Fitted on training data only')
print('  - Test data transformed using training statistics (no data leakage)')
print(f'  - Train mean: {X_train_scaled.mean().mean():.6f} (should be ~0)')
print(f'  - Train std:  {X_train_scaled.std().mean():.6f} (should be ~1)')

---
## 7. Handling Class Imbalance with SMOTE

SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic samples for the minority class (fraud) by interpolating between existing fraud samples. This is applied **only on training data** to avoid data leakage.

In [ ]:
# Apply SMOTE on training data ONLY
print('Before SMOTE:')
print(f'  Legitimate: {(y_train == 0).sum():,}')
print(f'  Fraud:      {(y_train == 1).sum():,}')

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f'\nAfter SMOTE:')
print(f'  Legitimate: {(y_train_smote == 0).sum():,}')
print(f'  Fraud:      {(y_train_smote == 1).sum():,}')
print(f'  Synthetic fraud samples added: {(y_train_smote == 1).sum() - (y_train == 1).sum():,}')
print(f'\nTest data unchanged: {len(X_test_scaled):,} samples')

In [ ]:
# Visualise the effect of SMOTE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before
before = [y_train.value_counts()[0], y_train.value_counts()[1]]
bars1 = axes[0].bar(['Legitimate', 'Fraud'], before, color=colors, edgecolor='black')
axes[0].set_title('Before SMOTE', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for bar, c in zip(bars1, before):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{c:,}', ha='center', va='bottom', fontweight='bold')

# After
after = [y_train_smote.value_counts()[0], y_train_smote.value_counts()[1]]
bars2 = axes[1].bar(['Legitimate', 'Fraud'], after, color=colors, edgecolor='black')
axes[1].set_title('After SMOTE', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')
for bar, c in zip(bars2, after):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{c:,}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Effect of SMOTE on Class Balance', fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

---
## 8. Clustering Analysis

In [ ]:
# K-Means: find optimal K using elbow method and silhouette score
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_2d)  # Using PCA-reduced data from earlier
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_2d, labels, sample_size=10000, random_state=RANDOM_STATE))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')

axes[1].plot(K_range, silhouettes, 'ro-')
axes[1].set_title('Silhouette Score', fontsize=14, fontweight='bold')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Score')

plt.tight_layout()
plt.show()

best_k = list(K_range)[np.argmax(silhouettes)]
print(f'Best K: {best_k}')

In [ ]:
# K-Means clustering and fraud analysis
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(X_2d)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cluster plot
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels, cmap='tab10', alpha=0.1, s=3)
axes[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c='red', marker='X', s=200, edgecolors='black', linewidths=2)
axes[0].set_title(f'K-Means (K={best_k})', fontsize=14, fontweight='bold')

# Fraud rate per cluster
cluster_df = pd.DataFrame({'Cluster': cluster_labels, 'Fraud': df['Class'].values})
fraud_rate = cluster_df.groupby('Cluster')['Fraud'].mean() * 100
axes[1].bar(fraud_rate.index, fraud_rate.values, color='#e74c3c', edgecolor='black')
axes[1].set_title('Fraud Rate per Cluster', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Fraud Rate (%)')

plt.tight_layout()
plt.show()

print('Fraud rate by cluster:')
for c, rate in fraud_rate.items():
    print(f'  Cluster {c}: {rate:.3f}%')

In [ ]:
# Isolation Forest - unsupervised anomaly detection
iso_forest = IsolationForest(n_estimators=200, contamination=0.00172, random_state=RANDOM_STATE, n_jobs=-1)
iso_pred = iso_forest.fit_predict(X_2d)
iso_pred_binary = (iso_pred == -1).astype(int)  # -1 = anomaly -> 1

print('Isolation Forest as fraud detector:')
print(f'  Anomalies detected: {iso_pred_binary.sum():,}')
print(f'  Actual frauds:      {df["Class"].sum():,}')
print(f'  Precision: {precision_score(df["Class"], iso_pred_binary):.4f}')
print(f'  Recall:    {recall_score(df["Class"], iso_pred_binary):.4f}')
print(f'  F1-Score:  {f1_score(df["Class"], iso_pred_binary):.4f}')

In [ ]:
# DBSCAN - density-based clustering
# Use a sample since DBSCAN is memory-intensive on large datasets
sample_size = 20000
sample_idx = np.random.choice(len(X_2d), sample_size, replace=False)

dbscan = DBSCAN(eps=0.8, min_samples=10)
dbscan_labels = dbscan.fit_predict(X_2d[sample_idx])

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = (dbscan_labels == -1).sum()

plt.figure(figsize=(10, 7))
plt.scatter(X_2d[sample_idx, 0], X_2d[sample_idx, 1], c=dbscan_labels, cmap='tab20', alpha=0.5, s=5)
plt.title(f'DBSCAN: {n_clusters} clusters, {n_noise:,} noise points', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Check if noise points contain more fraud
noise_fraud = df['Class'].values[sample_idx][dbscan_labels == -1].mean() * 100 if n_noise > 0 else 0
overall_fraud = df['Class'].mean() * 100
print(f'Fraud rate in noise points: {noise_fraud:.2f}% (vs overall: {overall_fraud:.3f}%)')

---
## 9. Model Training & Comparison

All models are trained on SMOTE-resampled training data and evaluated on the original, untouched test data.

In [ ]:
# Helper function to evaluate a model and store results
all_results = []
all_preds = {}
all_probs = {}

def evaluate(name, y_true, y_pred, y_prob=None, training_time=None):
    result = {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
    }
    if y_prob is not None:
        result['ROC-AUC'] = roc_auc_score(y_true, y_prob)
        result['AUPRC'] = average_precision_score(y_true, y_prob)
    if training_time:
        result['Time(s)'] = round(training_time, 2)

    all_results.append(result)
    all_preds[name] = y_pred
    if y_prob is not None:
        all_probs[name] = y_prob

    print(f'{name}:')
    print(f'  Accuracy={result["Accuracy"]:.4f}  Precision={result["Precision"]:.4f}  '
          f'Recall={result["Recall"]:.4f}  F1={result["F1-Score"]:.4f}  MCC={result["MCC"]:.4f}')
    if y_prob is not None:
        print(f'  ROC-AUC={result["ROC-AUC"]:.4f}  AUPRC={result["AUPRC"]:.4f}')

print('Evaluation function ready.')

### 9A. Classical ML Models

In [ ]:
# Logistic Regression
start = time.time()
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)
lr.fit(X_train_smote, y_train_smote)
t = time.time() - start

evaluate('Logistic Regression', y_test, lr.predict(X_test_scaled), lr.predict_proba(X_test_scaled)[:, 1], t)

In [ ]:
# Decision Tree
start = time.time()
dt = DecisionTreeClassifier(max_depth=15, random_state=RANDOM_STATE)
dt.fit(X_train_smote, y_train_smote)
t = time.time() - start

evaluate('Decision Tree', y_test, dt.predict(X_test_scaled), dt.predict_proba(X_test_scaled)[:, 1], t)

In [ ]:
# Random Forest
start = time.time()
rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_smote, y_train_smote)
t = time.time() - start

evaluate('Random Forest', y_test, rf.predict(X_test_scaled), rf.predict_proba(X_test_scaled)[:, 1], t)

In [ ]:
# SVM - using a subset because SVM is slow on large datasets
sample_n = min(50000, len(X_train_smote))
idx = np.random.choice(len(X_train_smote), sample_n, replace=False)

start = time.time()
svm = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
svm.fit(X_train_smote.iloc[idx], y_train_smote.iloc[idx])
t = time.time() - start

evaluate('SVM', y_test, svm.predict(X_test_scaled), svm.predict_proba(X_test_scaled)[:, 1], t)
print(f'  (Trained on {sample_n:,} samples for speed)')

In [ ]:
# KNN
start = time.time()
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_smote, y_train_smote)
t = time.time() - start

evaluate('KNN', y_test, knn.predict(X_test_scaled), knn.predict_proba(X_test_scaled)[:, 1], t)

### 9B. Advanced Ensemble Models

In [ ]:
# XGBoost
start = time.time()
xgb = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=RANDOM_STATE,
                    eval_metric='logloss', n_jobs=-1, use_label_encoder=False)
xgb.fit(X_train_smote, y_train_smote, eval_set=[(X_test_scaled, y_test)], verbose=False)
t = time.time() - start

evaluate('XGBoost', y_test, xgb.predict(X_test_scaled), xgb.predict_proba(X_test_scaled)[:, 1], t)

In [ ]:
# LightGBM
start = time.time()
lgbm = LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=RANDOM_STATE,
                      n_jobs=-1, verbose=-1)
lgbm.fit(X_train_smote, y_train_smote)
t = time.time() - start

evaluate('LightGBM', y_test, lgbm.predict(X_test_scaled), lgbm.predict_proba(X_test_scaled)[:, 1], t)

### 9C. Deep Learning Models

In [ ]:
# Neural Network (MLP)
n_features = X_train_smote.shape[1]

mlp = Sequential([
    Dense(128, activation='relu', input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

mlp.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

start = time.time()
history = mlp.fit(X_train_smote, y_train_smote, epochs=50, batch_size=512,
                  validation_data=(X_test_scaled, y_test),
                  callbacks=[EarlyStopping(patience=10, restore_best_weights=True)], verbose=0)
t = time.time() - start

mlp_prob = mlp.predict(X_test_scaled, verbose=0).flatten()
mlp_pred = (mlp_prob >= 0.5).astype(int)
evaluate('Neural Network', y_test, mlp_pred, mlp_prob, t)

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss Curve', fontweight='bold')
axes[0].legend()
axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Val')
axes[1].set_title('Accuracy Curve', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Autoencoder - trained on legitimate transactions only, detects fraud as anomalies
X_train_legit = X_train_scaled[y_train == 0]  # only normal transactions

ae_input = Input(shape=(n_features,))
encoded = Dense(64, activation='relu')(ae_input)
encoded = Dense(32, activation='relu')(encoded)
encoded = Dense(14, activation='relu')(encoded)  # bottleneck
decoded = Dense(32, activation='relu')(encoded)
decoded = Dense(64, activation='relu')(decoded)
decoded = Dense(n_features, activation='linear')(decoded)

autoencoder = Model(ae_input, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

start = time.time()
autoencoder.fit(X_train_legit, X_train_legit, epochs=50, batch_size=512,
                validation_split=0.1,
                callbacks=[EarlyStopping(patience=10, restore_best_weights=True)], verbose=0)
t = time.time() - start

# Reconstruction error = anomaly score
reconstructed = autoencoder.predict(X_test_scaled, verbose=0)
recon_error = np.mean((X_test_scaled - reconstructed) ** 2, axis=1)

# Set threshold based on training data
train_recon = autoencoder.predict(X_train_legit, verbose=0)
train_error = np.mean((X_train_legit - train_recon) ** 2, axis=1)
threshold = np.percentile(train_error, 97)

ae_pred = (recon_error > threshold).astype(int)
ae_prob = recon_error / recon_error.max()
evaluate('Autoencoder', y_test, ae_pred, ae_prob, t)

# Plot reconstruction error
plt.figure(figsize=(10, 5))
plt.hist(recon_error[y_test == 0], bins=100, alpha=0.7, color='#2ecc71', label='Legitimate', density=True)
plt.hist(recon_error[y_test == 1], bins=50, alpha=0.7, color='#e74c3c', label='Fraud', density=True)
plt.axvline(threshold, color='black', linestyle='--', label=f'Threshold={threshold:.4f}')
plt.title('Autoencoder Reconstruction Error', fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# LSTM - treats features as a sequence
X_train_seq = np.array(X_train_smote).reshape(-1, n_features, 1)
X_test_seq = np.array(X_test_scaled).reshape(-1, n_features, 1)

lstm = Sequential([
    LSTM(64, input_shape=(n_features, 1), return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

start = time.time()
lstm.fit(X_train_seq, y_train_smote, epochs=30, batch_size=1024,
         validation_data=(X_test_seq, y_test),
         callbacks=[EarlyStopping(patience=5, restore_best_weights=True)], verbose=0)
t = time.time() - start

lstm_prob = lstm.predict(X_test_seq, verbose=0).flatten()
lstm_pred = (lstm_prob >= 0.5).astype(int)
evaluate('LSTM', y_test, lstm_pred, lstm_prob, t)

In [ ]:
# 1D-CNN
cnn = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(n_features, 1), padding='same'),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.3),
    Conv1D(32, kernel_size=3, activation='relu', padding='same'),
    MaxPooling1D(2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

cnn.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

start = time.time()
cnn.fit(X_train_seq, y_train_smote, epochs=30, batch_size=1024,
        validation_data=(X_test_seq, y_test),
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)], verbose=0)
t = time.time() - start

cnn_prob = cnn.predict(X_test_seq, verbose=0).flatten()
cnn_pred = (cnn_prob >= 0.5).astype(int)
evaluate('1D-CNN', y_test, cnn_pred, cnn_prob, t)

### 9D. Hyperparameter Tuning (Optuna)

In [ ]:
# Tune XGBoost
print('Tuning XGBoost with Optuna (30 trials)...')

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    model = XGBClassifier(**params, random_state=RANDOM_STATE, eval_metric='logloss', n_jobs=-1, use_label_encoder=False)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(model, X_train_smote, y_train_smote, cv=cv, scoring='f1', n_jobs=-1)
    return scores.mean()

xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=30, show_progress_bar=True)
print(f'Best CV F1: {xgb_study.best_value:.4f}')

In [ ]:
# Train tuned XGBoost
start = time.time()
xgb_tuned = XGBClassifier(**xgb_study.best_params, random_state=RANDOM_STATE,
                          eval_metric='logloss', n_jobs=-1, use_label_encoder=False)
xgb_tuned.fit(X_train_smote, y_train_smote)
t = time.time() - start

evaluate('XGBoost (Tuned)', y_test, xgb_tuned.predict(X_test_scaled),
         xgb_tuned.predict_proba(X_test_scaled)[:, 1], t)

In [ ]:
# Tune LightGBM
print('Tuning LightGBM with Optuna (30 trials)...')

def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
    }
    model = LGBMClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(model, X_train_smote, y_train_smote, cv=cv, scoring='f1', n_jobs=-1)
    return scores.mean()

lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=30, show_progress_bar=True)
print(f'Best CV F1: {lgbm_study.best_value:.4f}')

In [ ]:
# Train tuned LightGBM
start = time.time()
lgbm_tuned = LGBMClassifier(**lgbm_study.best_params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
lgbm_tuned.fit(X_train_smote, y_train_smote)
t = time.time() - start

evaluate('LightGBM (Tuned)', y_test, lgbm_tuned.predict(X_test_scaled),
         lgbm_tuned.predict_proba(X_test_scaled)[:, 1], t)

---
## 10. Evaluation & Results

In [ ]:
# Model comparison table
results_df = pd.DataFrame(all_results).sort_values('F1-Score', ascending=False).reset_index(drop=True)

print('MODEL COMPARISON (sorted by F1-Score)')
print('=' * 100)
display(results_df)

best_model = results_df.iloc[0]['Model']
print(f'\nBest model: {best_model}')

In [ ]:
# Confusion matrices for all models
n_models = len(all_preds)
n_cols = 4
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

for i, (name, preds) in enumerate(all_preds.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

for i in range(n_models, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Confusion Matrices', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves
plt.figure(figsize=(12, 8))
for name, probs in all_probs.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc(fpr, tpr):.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves', fontsize=16, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall curves
plt.figure(figsize=(12, 8))
for name, probs in all_probs.items():
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    plt.plot(rec, prec, linewidth=2, label=f'{name} (AP={ap:.4f})')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves', fontsize=16, fontweight='bold')
plt.legend(loc='upper right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Model comparison bar chart
metrics_plot = ['Precision', 'Recall', 'F1-Score', 'MCC']
plot_data = results_df[['Model'] + metrics_plot].set_index('Model')

plot_data.plot(kind='bar', figsize=(16, 8), width=0.8, edgecolor='black')
plt.title('Model Comparison', fontsize=16, fontweight='bold')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='lower right')
plt.ylim(0, 1.1)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP explainability for the best tree-based model
print(f'SHAP Analysis for best model...')

# Use XGBoost tuned for SHAP
shap_model = xgb_tuned
shap_sample = X_test_scaled.sample(min(2000, len(X_test_scaled)), random_state=RANDOM_STATE)

explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(shap_sample)

# Feature importance
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, shap_sample, plot_type='bar', show=False, max_display=15)
plt.title('SHAP Feature Importance', fontweight='bold')
plt.tight_layout()
plt.show()

# Detailed impact
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, shap_sample, show=False, max_display=15)
plt.title('SHAP Feature Impact', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Classification report for best model
best_preds = all_preds.get(best_model, all_preds['XGBoost (Tuned)'])
print(f'Classification Report - {best_model}')
print('=' * 60)
print(classification_report(y_test, best_preds, target_names=['Legitimate', 'Fraud']))

In [ ]:
# Final Summary
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)

print(f'\nDataset: {df.shape[0]:,} transactions, {(df["Class"]==1).sum()} fraud ({df["Class"].mean()*100:.3f}%)')
print(f'Models trained: {len(all_results)}')
print(f'Best model: {best_model}')

best = results_df.iloc[0]
print(f'\nBest model metrics:')
print(f'  Precision: {best["Precision"]:.4f}')
print(f'  Recall:    {best["Recall"]:.4f}')
print(f'  F1-Score:  {best["F1-Score"]:.4f}')

print(f'\nKey findings:')
print(f'  - Accuracy alone is misleading for imbalanced data')
print(f'  - Recall is critical: catching fraud matters more than avoiding false alarms')
print(f'  - SMOTE on training data only prevents data leakage')
print(f'  - Ensemble models (XGBoost/LightGBM) perform best on tabular data')

print(f'\nFuture work:')
print(f'  - Deploy as a REST API for real-time predictions')
print(f'  - Explore federated learning for privacy-preserving training')
print(f'  - Implement real-time monitoring with model drift detection')